# 01 - Netflix Prize Data Loading and Preprocessing

**Person A's Responsibility**: Build a shared data loading pipeline for the team that produces `ratings.parquet` for subsequent EDA, recommendation systems, clustering, and RFM analysis.

## Data Sources
- `dataset/training_set/`: 17,770 `mv_*.txt` files, each containing the ratings for a single movie
- `dataset/movie_titles.txt`: Movie ID, release year, and title
- `dataset/probe.txt`: Test set (used for evaluating recommendation models)

## 1. Import libraries and set up paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

# Development mode: Set to 0.1 to load only 10% of movies for faster testing; set to 1.0 to load the full dataset (recommended for Phase 1)
SAMPLE_FRACTION = 1.0

# Set project root: If running in notebooks/, go up one level to look for dataset/
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if (cwd.parent / "dataset").exists() else cwd
DATA_DIR = PROJECT_ROOT / "dataset"
TRAINING_DIR = DATA_DIR / "training_set"
OUTPUT_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Project root directory: {PROJECT_ROOT}")
print(f"Training set directory: {TRAINING_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

Project root directory: /Users/wuyusen/Desktop/Northwestern MLDS/421 Data Mining/final project
Training set directory: /Users/wuyusen/Desktop/Northwestern MLDS/421 Data Mining/final project/dataset/training_set
Output directory: /Users/wuyusen/Desktop/Northwestern MLDS/421 Data Mining/final project/data


## 2. Parse movie_titles.txt

Format: `MovieID,YearOfRelease,Title` (Note: Title may contain commas, so use `split(',', 2)` to only split at the first two commas)

In [2]:
def load_movie_titles(filepath: Path) -> pd.DataFrame:
    """
    Parse movie_titles.txt and produce a DataFrame containing MovieID, YearOfRelease, Title.

    Args:
        filepath: Path to movie_titles.txt

    Returns:
        DataFrame with three columns: MovieID (int), YearOfRelease (int), Title (str)

    Note: Use split(',', 2) because the Title field may contain commas (e.g., "Lord of the Rings: ...")
    """
    rows = []
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",", 2)  # Split at most 2 times; the rest is Title
            if len(parts) >= 3:
                movie_id = int(parts[0])
                # YearOfRelease may be "NULL" (some movies do not have a release year), set to None if not a digit
                year = int(parts[1]) if parts[1].isdigit() else None
                title = parts[2].strip()
                rows.append({"MovieID": movie_id, "YearOfRelease": year, "Title": title})
            elif len(parts) == 2:
                rows.append({"MovieID": int(parts[0]), "YearOfRelease": int(parts[1]) if parts[1].isdigit() else None, "Title": ""})
    return pd.DataFrame(rows)

movie_titles = load_movie_titles(DATA_DIR / "movie_titles.txt")
print(f"Number of movies: {len(movie_titles)}")
movie_titles.head(10)

Number of movies: 17770


,MovieID,YearOfRelease,Title
0,1,2003.0,Dinosaur Planet
1,2,2004.0,Isle of Man TT 2004 Review
2,3,1997.0,Character
3,4,1994.0,Paula Abdul's Get Up & Dance
4,5,2004.0,The Rise and Fall of ECW
5,6,1997.0,Sick
6,7,1992.0,8 Man
7,8,2004.0,What the #$*! Do We Know!?
8,9,1991.0,Class of Nuke 'Em High 2
9,10,2001.0,Fighter


## 3. Parse all mv_*.txt files in training_set
Each file format:
- First line: `MovieID:` (e.g., `1:`)
- Subsequent lines: `CustomerID,Rating,Date` (e.g., `1488844,3,2005-09-06`)

In [3]:
def parse_single_movie_file(filepath: Path) -> list[dict]:
    """
    Parse a single movie ratings file and return all ratings for that movie as a list of dicts.

    Args:
        filepath: Path to mv_XXXXXXX.txt
    
    Returns:
        [{"MovieID": int, "CustomerID": int, "Rating": int, "Date": str}, ...]
    
    Note: The filename mv_0000001.txt corresponds to MovieID=1, and the first line "1:" also indicates the MovieID.
    """
    rows = []
    movie_id = None
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.endswith(":"):
                movie_id = int(line[:-1])
                continue
            parts = line.split(",")
            if len(parts) >= 3 and movie_id is not None:
                try:
                    customer_id = int(parts[0])
                    rating = int(parts[1])
                    date_str = parts[2]
                    rows.append({
                        "MovieID": movie_id,
                        "CustomerID": customer_id,
                        "Rating": rating,
                        "Date": date_str
                    })
                except (ValueError, IndexError):
                    pass
    return rows


def load_all_ratings(training_dir: Path) -> pd.DataFrame:
    """
    Iterate through all mv_*.txt files in training_set and combine them into a single DataFrame.

    Args:
        training_dir: Path to the training_set directory

    Returns:
        DataFrame with columns: MovieID, CustomerID, Rating, Date

    Note: Uses tqdm to display progress, as there are many files (17,770).
    """
    movie_files = sorted(training_dir.glob("mv_*.txt"))
    if SAMPLE_FRACTION < 1.0:
        np.random.seed(42)
        n_sample = max(1, int(len(movie_files) * SAMPLE_FRACTION))
        movie_files = np.random.choice(movie_files, n_sample, replace=False).tolist()
        print(f"Development mode: only loading {n_sample} files ({SAMPLE_FRACTION*100:.0f}%)")
    print(f"Found {len(movie_files)} movie rating files")
    
    all_rows = []
    for fp in tqdm(movie_files, desc="Parsing rating files"):
        rows = parse_single_movie_file(fp)
        all_rows.extend(rows)
    
    df = pd.DataFrame(all_rows)
    return df

# Run loading (may take several minutes)
ratings_raw = load_all_ratings(TRAINING_DIR)
print(f"\nTotal number of ratings: {len(ratings_raw):,}")
ratings_raw.head()

Found 17770 movie rating files


Parsing rating files: 100%|██████████| 17770/17770 [00:58<00:00, 302.35it/s]



Total number of ratings: 100,480,507


,MovieID,CustomerID,Rating,Date
0,1,1488844,3,2005-09-06
1,1,822109,5,2005-05-13
2,1,885013,4,2005-10-19
3,1,30878,4,2005-12-26
4,1,823519,3,2004-05-03


## 4. Merge Movie Information and Extract Temporal Features
- Merge `ratings` and `movie_titles` on MovieID to obtain YearOfRelease, Title
- Convert Date to datetime and extract Year, Month, DayOfWeek

In [4]:
# Merge movie titles and year of release
ratings = ratings_raw.merge(movie_titles, on="MovieID", how="left")

# Convert Date string to datetime
ratings["Date"] = pd.to_datetime(ratings["Date"])

# Extract temporal features: year, month, day of week of each rating
ratings["Year"] = ratings["Date"].dt.year
ratings["Month"] = ratings["Date"].dt.month
ratings["DayOfWeek"] = ratings["Date"].dt.dayofweek  # 0=Monday, 6=Sunday

print("Columns:", ratings.columns.tolist())
print("\nFirst 5 rows:")
ratings.head()

Columns: ['MovieID', 'CustomerID', 'Rating', 'Date', 'YearOfRelease', 'Title', 'Year', 'Month', 'DayOfWeek']

First 5 rows:


,MovieID,CustomerID,Rating,Date,YearOfRelease,Title,Year,Month,DayOfWeek
0,1,1488844,3,2005-09-06,2003.0,Dinosaur Planet,2005,9,1
1,1,822109,5,2005-05-13,2003.0,Dinosaur Planet,2005,5,4
2,1,885013,4,2005-10-19,2003.0,Dinosaur Planet,2005,10,2
3,1,30878,4,2005-12-26,2003.0,Dinosaur Planet,2005,12,0
4,1,823519,3,2004-05-03,2003.0,Dinosaur Planet,2004,5,0


## 5. Save as ratings.parquet
Parquet is a columnar storage format that is fast to read/write and highly compressed, making it well-suited for large DataFrames.

In [5]:
output_path = OUTPUT_DIR / "ratings.parquet"
ratings.to_parquet(output_path, index=False)
print(f"Saved to {output_path}")
print(f"File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

Saved to /Users/wuyusen/Desktop/Northwestern MLDS/421 Data Mining/final project/data/ratings.parquet
File size: 756.44 MB


## 6. Parse probe.txt
probe.txt format: a subset of (MovieID, CustomerID) pairs for evaluating the recommendation model.
- Each section starts with `MovieID:`
- Each following line is a CustomerID (users to be predicted for that movie in probe)

In [6]:
def load_probe(filepath: Path) -> pd.DataFrame:
    """
    Parse probe.txt and generate a DataFrame of (MovieID, CustomerID) pairs.

    Args:
        filepath: Path to probe.txt

    Returns:
        DataFrame containing MovieID and CustomerID

    Note: probe is a Netflix-supplied test subset, used for calculating RMSE and similar metrics.
    """
    rows = []
    movie_id = None
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.endswith(":"):
                movie_id = int(line[:-1])
                continue
            if movie_id is not None:
                try:
                    customer_id = int(line)
                    rows.append({"MovieID": movie_id, "CustomerID": customer_id})
                except ValueError:
                    pass
    return pd.DataFrame(rows)

probe = load_probe(DATA_DIR / "probe.txt")
print(f"Number of probe records: {len(probe):,}")
probe.head(10)

Number of probe records: 1,408,395


,MovieID,CustomerID
0,1,30878
1,1,2647871
2,1,1283744
3,1,2488120
4,1,317050
5,1,1904905
6,1,1989766
7,1,14756
8,1,1027056
9,1,1149588


In [7]:
# Save probe as parquet
probe_path = OUTPUT_DIR / "probe.parquet"
probe.to_parquet(probe_path, index=False)
print(f"Saved to {probe_path}")

Saved to /Users/wuyusen/Desktop/Northwestern MLDS/421 Data Mining/final project/data/probe.parquet


## 7. Validation and Summary
Verify data integrity for use by the entire team.

In [8]:
print("=== Ratings Summary ===")
print(ratings.info())
print("\nRating Distribution:")
print(ratings["Rating"].value_counts().sort_index())
print("\nDate Range:", ratings["Date"].min(), "~", ratings["Date"].max())
print("\n=== Done ===")
print("Output files:")
print(f"  - {OUTPUT_DIR / 'ratings.parquet'}")
print(f"  - {OUTPUT_DIR / 'probe.parquet'}")

=== Ratings Summary ===
<class 'pandas.DataFrame'>
RangeIndex: 100480507 entries, 0 to 100480506
Data columns (total 9 columns):
 #   Column         Dtype         
---  ------         -----         
 0   MovieID        int64         
 1   CustomerID     int64         
 2   Rating         int64         
 3   Date           datetime64[us]
 4   YearOfRelease  float64       
 5   Title          str           
 6   Year           int32         
 7   Month          int32         
 8   DayOfWeek      int32         
dtypes: datetime64[us](1), float64(1), int32(3), int64(3), str(1)
memory usage: 7.2 GB
None

Rating Distribution:
Rating
1     4617990
2    10132080
3    28811247
4    33750958
5    23168232
Name: count, dtype: int64

Date Range: 1999-11-11 00:00:00 ~ 2005-12-31 00:00:00

=== Done ===
Output files:
  - /Users/wuyusen/Desktop/Northwestern MLDS/421 Data Mining/final project/data/ratings.parquet
  - /Users/wuyusen/Desktop/Northwestern MLDS/421 Data Mining/final project/data/probe.parq